# Phase 2 — Supervised baselines

Run `python scripts/phase_02_acceptance.py` first. Reusable training and metrics live in `pipelines/train.py` and `evaluation/`; notebooks read source/config/code-verified aggregate outputs. Dummy prior, logistic and balanced logistic share the exact primary split. AP is sklearn average precision, not trapezoidal PR area. Validation performance is model-selection evidence, not final test performance.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from fraud_detection.paths import find_project_root
from fraud_detection.pipelines.train import load_verified_results

ROOT = find_project_root()
OUTPUT = ROOT / 'artifacts/phase_02'
result = load_verified_results(ROOT)
print('Source kind:', result['source_kind'])
print('Config SHA:', result['config_sha256'])
print('Test opened:', result['test_opened'])


In [ ]:
names = ['dummy_prior', 'logistic', 'logistic_balanced']
baselines = [run for run in result['runs'] if run['model_name'] in names]
table = pd.DataFrame(result['comparison'])
display(table[table['model'].isin(names)])
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for run in baselines:
    curve = run['curves']
    axes[0].plot(curve['recall'], curve['precision'], label=run['model_name'])
    axes[1].plot(curve['fpr'], curve['tpr'], label=run['model_name'])
axes[0].set(xlabel='Recall', ylabel='Precision', title='Validation PR curves')
axes[1].set(xlabel='False-positive rate', ylabel='True-positive rate', title='Validation ROC curves')
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()


## Interpretation

A high majority-only accuracy does not establish fraud detection quality. Compare AP first, then recall, precision, false alerts and missed fraud at each validation-selected operating point. Class weights are learned from training labels. Test remains sealed.